In [1]:
print("Hello, Chris and Daliah!")

Hello, Chris and Daliah!


In [2]:
print("Hello, Chris and Daliah!, testing functional branch") #comment test

Hello, Chris and Daliah!, testing functional branch


# Task 2: Running an Original eLCS on the Raw SUPPORT2 dataset.
* Select an appropriate LCS variant, preferably eLCS.
* Use the original LCS code without algorithmic modification.
* Run the original LCS system on the raw or minimally processed dataset.
* Report the selected LCS parameters.
* Record the baseline performance results.
* Explain any minimal processing required to make the dataset compatible with the LCS code.

For Jono and Daaliah: pip install scikit-elcs run this in your terminal to get the eLCS, same one from LCS labs. 

In [3]:
import numpy as np
import pandas as pd

from skeLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import inspect

print("eLCS loaded from:")
print(inspect.getfile(eLCS))

eLCS loaded from:
c:\Users\itsjo\AppData\Local\Programs\Python\Python314\Lib\site-packages\skeLCS\eLCS.py


In [4]:
df_raw = pd.read_csv("support2.csv")

print(df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes.value_counts()) 
# loading the dataset and displaying its shape, first few rows, and data types of each column.
# this is teh raw dataset df_raw

df_baseline = df_raw.copy(deep=True) 
# preservation of the original dataset for baseline comparison. 
# task 2 is the baseline which will be used to compare the performance of the eLCS model later on. 

(9105, 47)


,age,death,sex,hospdead,slos,d.time,dzgroup,dzclass,num.co,edu,...,crea,sod,ph,glucose,bun,urine,adlp,adls,sfdm2,adlsc
1,62.84998,0,male,0,5,2029,Lung Cancer,Cancer,0,11.0,...,1.199951,141.0,7.459961,NaN,NaN,NaN,7.0,7.0,NaN,7.0
2,60.33899,1,female,1,4,4,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,5.500000,132.0,7.250000,NaN,NaN,NaN,NaN,1.0,<2 mo. follow-up,1.0
3,52.74698,1,female,0,17,47,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,2.000000,134.0,7.459961,NaN,NaN,NaN,1.0,0.0,<2 mo. follow-up,0.0
4,42.38498,1,female,0,3,133,Lung Cancer,Cancer,2,11.0,...,0.799927,139.0,NaN,NaN,NaN,NaN,0.0,0.0,no(M2 and SIP pres),0.0
5,79.88495,0,female,0,16,2029,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,...,0.799927,143.0,7.509766,NaN,NaN,NaN,NaN,2.0,no(M2 and SIP pres),2.0


float64    31
int64       8
str         8
Name: count, dtype: int64


In [5]:
target = "sfdm2" # target variable for the classification task, which is the column 'sfdm2' in the dataset.

print(df_baseline[target].value_counts(dropna=False)) # inspection of the target variable. 

# incomplete values are expected in df_raw. 

df_baseline = df_baseline.dropna(subset=[target]).copy()
print("Rows after removing missing target:", len(df_baseline))
# because of the way eLCS works, the taget variables must not contain "NaN", only numerical values so this will 
# be an example of minimal preprocessing. 
# a supervised learning algorithm like eLCS requires a complete target
#  variable to learn from the data, which is why we have dropped na. 

sfdm2
<2 mo. follow-up       3123
no(M2 and SIP pres)    3061
NaN                    1400
adl>=4 (>=5 if sur)     916
SIP>=30                 564
Coma or Intub            41
Name: count, dtype: int64
Rows after removing missing target: 7705


In [6]:
# another thing is that scikit-eLCS supports continuous and discrete attributes, mixed feature types and 
# missing predictor values, but its core estimator expects the data passed into fit() to be numeric. 

# so strings will have to be encoded into numbers. sorry jono ily <3
X_df = df_baseline.drop(columns=[target]).copy()
y_raw = df_baseline[target].copy()

categorical_cols = X_df.select_dtypes(
    include=["object", "string", "category"]
).columns

print("Categorical predictors:")
print(categorical_cols.tolist()) # identify categorical columns in the dataset.

category_mappings = {}

for col in categorical_cols:
    codes, labels = pd.factorize(X_df[col], sort=True)

    X_df[col] = codes.astype(float)

    # factorize represents missing values as -1.
    # Put these back to NaN because eLCS supports missing predictor values.
    X_df.loc[X_df[col] == -1, col] = np.nan

    category_mappings[col] = labels.tolist()

    # numbers do not have hierarchy, 3 > 2 is not true. 

Categorical predictors:
['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr']


In [7]:
# the target variable is also categorical, so it will be encoded as well.
# but separately. 

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y_raw)

X = X_df.to_numpy(dtype=float)

print("Target mapping:") # so we know what the encoded values mean.

for i, class_name in enumerate(target_encoder.classes_):
    print(i, "=", class_name)

#data check. 
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Missing predictor values:", np.isnan(X).sum())
print("Missing target values:", pd.isna(y).sum())
# Predictor missingness doesn't matter, eLCS can handle that naturally. 
# target missingness is 0, good. eLCS requires a complete target variable to learn from the data.

Target mapping:
0 = <2 mo. follow-up
1 = Coma or Intub
2 = SIP>=30
3 = adl>=4 (>=5 if sur)
4 = no(M2 and SIP pres)
X shape: (7705, 46)
y shape: (7705,)
Missing predictor values: 36948
Missing target values: 0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
) # test train split, 80% training, 20% testing, this matches the later split as well so might as well use the same split. 

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# stratify=y? sfdm2 is imbalanced, so stratifying the split ensures that both the training and test sets have a similar 
# distribution of the target classes. This is important for model evaluation, as it helps to avoid bias in the performance 
# metrics due to class imbalance.

print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Training set: (6164, 46)
Test set: (1541, 46)
Training class distribution:
0    0.405256
1    0.005354
2    0.073167
3    0.118916
4    0.397307
Name: proportion, dtype: float64

Testing class distribution:
0    0.405581
1    0.005191
2    0.073329
3    0.118754
4    0.397145
Name: proportion, dtype: float64


In [9]:
# implementation beginss. 
baseline_elcs = eLCS(
    random_state=42
)
# this is our baseline model, with default parameters everywhere, those parameters is what we will be tuning
# once we move further into the tasks. 
# below are the default values from scikit-eLCS. 
#learning_iterations = 10000
#N = 1000
#p_spec = 0.5
#nu = 5
#chi = 0.8
#mu = 0.04
#theta_GA = 25
#selection_method = tournament

params = baseline_elcs.get_params()

params_df = pd.DataFrame(
    params.items(),
    columns=["Parameter", "Value"]
)

display(params_df) # this function displays the parameters of the baseline eLCS model in a tabular format,
#making it easier to review and understand the configuration of the model before training.

# gonna save it to a csv for later reference.
# this also makes reporting easier. 
params_df.to_csv(
    "task2_original_elcs_parameters.csv",
    index=False
)

,Parameter,Value
0,N,1000
1,acc_sub,0.99
2,beta,0.2
3,chi,0.8
4,delta,0.1
5,discrete_attribute_limit,10
6,do_GA_subsumption,True
7,do_correct_set_subsumption,False
8,fitness_reduction,0.1
9,init_fit,0.01


In [10]:
baseline_elcs.fit(X_train, y_train); 
print("eLCS training complete.")

# do not remove the ; idky but the scikit-eLCS output is such that jupyter clashes with it,
# so it will train perfectly fine but just wont show you nicely. ; makes it so that the chunk runs but doesnt display. 
# we can view it later= 

# boom thats the whole thing. thats our model. 
# whats actually happening in here tho? its happening in order, 1 > 2 > 3

# 1. take current patient (instance) - dont lose track of the current patient,
# 2. construct match set [M], match set is the collection of classifiers that match the current instance based on their conditions. 
# 3. construct correct set [C], the subset of the match set that contains classifiers that correctly classify the current instance.
# 4. update rule parameters/fitness 

# 5. maybe performs subsumption, which is the process of replacing a more specific classifier with a more 
# general one if the general one has better performance.

# 6. maybe performs GA genetic algorithm, which is a search heuristic that mimics the process of natural selection to
# generate high-quality solutions for optimization and search problems.

# 7. crossover/mutation generate rules. 
# 8. delete rules if the population size exceeds the maximum allowed size.
# 9. next patient (instance), repeats. each. time. 

# important note: learning_interations = 10000, means 10,000 individual learning cycles. 

eLCS training complete.


In [11]:
print("Training completed:", baseline_elcs.hasTrained)
print("Iterations completed:", baseline_elcs.explorIter)
print("Number of rules:", len(baseline_elcs.population.popSet))

Training completed: True
Iterations completed: 10000
Number of rules: 948


In [12]:
y_pred = baseline_elcs.predict(X_test)
# what is this? this is the prediction step, 
# where the trained eLCS model is used to predict the 
# target variable for the test set instances.

# an important note for later:
# baseline_elcs.score(X_test, y_test)
# the eLCS model overrides this and returns balanced score, not ordinary accuracy. 

# whats the point?
# baseline_elcs.score(xxxxx) should not be labeled as accuracy, because it is not. it is balanced accuracy.

accuracy = accuracy_score(y_test, y_pred)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred
)

precision_macro = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

# all of these are to be stored and saved. 
baseline_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Value": [
        accuracy,
        balanced_acc,
        precision_macro,
        recall_macro,
        f1_macro
    ]
})

display(baseline_results)

baseline_results.to_csv(
    "task2_original_elcs_results.csv",
    index=False
) # this is saved to a CSV aswell since everything we do later will be compared to this, nice to have a seprate 
# file for it, ya feel?

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    )
)

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=target_encoder.classes_,
    columns=target_encoder.classes_
)

display(cm_df)

,Metric,Value
0,Accuracy,0.718365
1,Balanced Accuracy,0.358928
2,Macro Precision,0.343498
3,Macro Recall,0.358928
4,Macro F1,0.322452


                     precision    recall  f1-score   support

   <2 mo. follow-up       0.82      0.85      0.83       625
      Coma or Intub       0.00      0.00      0.00         8
            SIP>=30       0.00      0.00      0.00       113
adl>=4 (>=5 if sur)       0.25      0.01      0.01       183
no(M2 and SIP pres)       0.65      0.94      0.77       612

           accuracy                           0.72      1541
          macro avg       0.34      0.36      0.32      1541
       weighted avg       0.62      0.72      0.64      1541



,<2 mo. follow-up,Coma or Intub,SIP>=30,adl>=4 (>=5 if sur),no(M2 and SIP pres)
<2 mo. follow-up,530,0,0,1,94
Coma or Intub,3,0,0,0,5
SIP>=30,19,0,0,2,92
adl>=4 (>=5 if sur),62,2,0,1,118
no(M2 and SIP pres),35,0,1,0,576


In [13]:
# super handy, we can export the learned rules. Pretty sure this pops up later so I think saving the 
# initial rules could be useful.
baseline_elcs.export_final_rule_population(
    headerNames=np.array(X_df.columns),
    className="sfdm2",
    filename="task2_original_elcs_rules.csv"
)

# and 

baseline_elcs.export_iteration_tracking_data(
    "task2_original_elcs_tracking.csv"
)

# theres an issue I wanna raise tho, this minimally processed dataset which the model has been trained on access to 
# the future variables... do we want that? or should future variable cleansing count as minimal preprocessing? 
# see what yall think. 

# > [Jono]: I doubt we want one model to have access to both so we should train a sperate one in task 4 imo

# Task 3: Data Preprocessing and Feature Engineering
* Address missing values, duplicate records, invalid values, inconsistent categories, and incorrect data types. 
* Investigate and handle outliers where appropriate. 
* Encode categorical variables and scale, discretise, or transform numerical variables where required. 
* Address class imbalance where relevant. 
* Apply feature engineering, feature selection, or dimensionality reduction where justified. 
* Clearly explain how preprocessing is expected to improve LCS performance. 
* Ensure that preprocessing avoids data leakage. 

For Chris and Daliah: pip install imblearn

In [14]:
# Create a new dataframe to clean

df_clean = df_raw.copy(deep=True)
df_clean.shape

(9105, 47)

In [15]:
# Filter Ages
df_clean = df_clean[(df_clean['age'] >= 0) & (df_clean['age'] <= 110)]

# Number of records removed
removed_records = df_raw.shape[0] - df_clean.shape[0]
print(f"Number of records removed due to invalid age: {removed_records}")

Number of records removed due to invalid age: 0


In [16]:
# Whether they died is irrelevant to this study

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['death', 'hospdead'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 2


In [17]:
# 'slos' denotes number of days they were in hospital before being discharged.
# A longer stay might imply they are more sickly and have worse odds once discharged.

# Number of records removed
removed_records = df_raw.shape[0] - df_clean.shape[0]
print(f"Number of records removed due to invalid value: {removed_records}")

Number of records removed due to invalid value: 0


In [18]:
# 'd.time' denotes number of days patients were in the study. 
# If they didn't reach 2 months they didn't get an sfdm2 value.
# If they exceeded it then they got one and the variable is no longer relevant.
# We can remove the feature

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['d.time'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 1


In [19]:
# 'dzgroup' and 'dzclass' and 'num.co' denote disease types and numbers which will be very helpful.
# Nothing looks out of place.

In [20]:
# Many education level entries are missing. There is also an individual income column.
# We assumption that education level will not directly affect deterioration in terminally ill patients
# The whole column will simply be removed as its both difficult to average and of low importance.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['edu'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 1


In [21]:
# Income feature is missing to many phenotypes

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['income'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 1


In [22]:
# 'scoma' is a measure of how deep in a coma they are on a scale of 0 - 100.
# All values are in valid ranges but one entry is missing.

before = df_clean.shape[0]
df_clean = df_clean[df_clean['scoma'].notna()]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 1


In [23]:
# 'charges' denotes their number of hospital charges.
# This could be useful but there are multiple missing entries

before = df_clean.shape[0]
df_clean = df_clean[df_clean['charges'].notna()]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 171


In [24]:
# 'totcst', 'totmcst' are cost related iteams that are missing to many values.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['totcst', 'totmcst'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 2


In [25]:
# 'avtisst' is a score related to amount of care required from days 3 to 25.
# This could be useful but there are multiple missing entries

before = df_clean.shape[0]
df_clean = df_clean[df_clean['avtisst'].notna()]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 81


In [26]:
# 'race' denotes as implied.
# Useful information, only a few entires missing, and looks clean based on TS1 scope.
# Will remove empty entries.

before = df_clean.shape[0]
df_clean = df_clean[df_clean['race'].notna()]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")
print(f"Remaining rows: {after}")

Rows removed: 42
Remaining rows: 8810


In [27]:
# 'sps' and 'aps' are both physiological scores taken at day 3 that may be very useful
# 'sps' ranges from 0 to 163 and 'aps' ranges from 0 to 299, meaning all recorded values are within spec according to scoping in task 1.
# A few records are missing

before = df_clean.shape[0]
df_clean = df_clean[df_clean['sps'].notna()]
df_clean = df_clean[df_clean['aps'].notna()]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

#These entries must have already been removed in a previous removal.

Rows removed: 0


In [28]:
# 'surv2m' and 'surv6m' is a machine's estimate of the patients odds of survival in 2 and 6 months time.
# This wont actually impact what condition they're in in 2 months so we will remove it entirely.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['surv2m', 'surv6m'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 2


In [29]:
# 'hday' denotes the number of days the patient had already been in hospital before being diagnosed as terminally ill.
# This is useful as it tells us how long we have known them to be in this state.
# No entries missing and seems fine based on T1 scope.

In [30]:
# 'diabetes' and 'dementia' are as named and have binary values.
# Useful information with no cleaning required.

In [31]:
# 'ca' denotes cancer and has three potential values.
# Useful information
# No entries missing and seems fine based on T1 scope.

In [32]:
# As per 'surv2m' and 'surv6m', 'prg2m' and 'prg6m' are practitioners guesses at the well being of the patient in 2 and 6 months time.
# Will not effect actual outcome so will be removed.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['prg2m', 'prg6m'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 2


In [33]:
# 'dnrday' marks day the DNR order was given.
# Our model is meant to function on information recieved in day three. Anything after this is seeing into the future.
# The large majority of instances have phenotypes within this feature exceeding 3 days, and therefore the feature is removed.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['dnrday'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 1


In [34]:
# 'dnr' has to be removed due to the removal of dnr day. The order was probably given much after the 3 day mark.
before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['dnr'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 1


In [35]:
# 'meanbp' is a blood pressure reading on day three. 
# Useful information but 1 record missing, and the minimum is recorded as 0, which is impossible.
# The maximum is very extreme, but these are terminally ill patients. 
# Extremely high and low blood pressures should not be instantly be deemed errors in this instance.

before = df_clean.shape[0]
df_clean = df_clean[(df_clean['meanbp'] != 0) & (df_clean['meanbp'].notna())]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 47


In [36]:
# 'wblc' is a white blood cell count measured in thousands on day 3.     
# Useful information but multiple records missing and more absurd values.
# There are records of very high white blood cell counts, but depending on the disease that might make sense.
# Further analysis showed that all cases of high wblc were tied to patients with diseaess knwon to cuase high wblc.
# This lead to the following action:

before = df_clean.shape[0]
df_clean = df_clean[(df_clean['wblc'] != 0) & (df_clean['wblc'].notna())]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 217


In [37]:
# 'hrt' refers to heart rate measured on day 3.
# Useful information but has readings from 0 to 300. Anything below 30 is practically implausible, so will be considered invalid.
# Will remove the NaN and <20 rows along with the 300 row before rechecking boundires.

before = df_clean.shape[0]

df_clean = df_clean[(df_clean['hrt'] >= 30) & (df_clean['hrt'].notna())]

# Remove the isolated 300 outlier
df_clean = df_clean[df_clean['hrt'] != 300]

after = df_clean.shape[0]
print(f"Rows removed: {before - after}")

# Recheck min and max
print("New min hrt:", df_clean['hrt'].min())
print("New max hrt:", df_clean['hrt'].max())

Rows removed: 48
New min hrt: 30.0
New max hrt: 250.0


In [38]:
# 'resp' is the resperation rate of the patient measured on day 3. 
# Useful information but one record marked as missing and more entries at 0.
# Any reocrds at a rate below 4 will be removed.
# 90 seems plausible so will be left as is. 

before = df_clean.shape[0]

df_clean = df_clean[df_clean['resp'] != 0]
df_clean = df_clean[df_clean['resp'] >= 4]

after = df_clean.shape[0]
print(f"Rows removed: {before - after}")
print(f"Remaining rows: {after}")

print("New min resp:", df_clean['resp'].min())
print("New max resp:", df_clean['resp'].max())

Rows removed: 18
Remaining rows: 8480
New min resp: 4.0
New max resp: 90.0


In [39]:
# 'temp' refers to body tempreture.
# Useful information that seems to land within bounds but is marked as missing one entry.

before = df_clean.shape[0]
df_clean = df_clean[(df_clean['temp'].notna())]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

# Must have been previously removed.

Rows removed: 0


In [40]:
# 'pafi', 'alb' and 'bili' are all simply missing to many rows to be used to have averages applied.
# All three were removed.

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['pafi', 'alb', 'bili'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")

Columns removed: 3


In [41]:
# 'crea' and 'sod' denote creatine and sodium levels measured on day 3. 
# Useful information with only a few missing entries.
# Unsure of what values here are reasonable but there are no zero entries so I will only remove NaN for now.

before = df_clean.shape[0]
df_clean = df_clean[(df_clean['crea'].notna())]
df_clean = df_clean[(df_clean['sod'].notna())]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 19


In [42]:
# Repeat of 'pafi', 'alb' and 'bili'
# 'ph', gluscose', 'bun', 'urine', 'adlp' and 'adls' all hope potentially useful information but are missing far too many records.
# All columns will be removed

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['ph', 'glucose', 'bun', 'urine', 'adlp', 'adls'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")    

Columns removed: 6


In [43]:
# 'sfdm2' is our target prediction feature. 
# Hence, while it is missing a lot of values, we can only use entires with this feature.
# All missing this feature will be removed.

before = df_clean.shape[0]
df_clean = df_clean[(df_clean['sfdm2'].notna())]
after = df_clean.shape[0]

print(f"Rows removed: {before - after}")

Rows removed: 1280


In [44]:
# 'adlsc' refers to a wellbeing measurement taken at the 2 month meetings that we are trying to predict.
# This means that it's data the mlm should not have access too when trying to train.
# Removing the column

before = df_clean.shape[1]
df_clean = df_clean.drop(columns=['adlsc'])
after = df_clean.shape[1]

print(f"Columns removed: {before - after}")   

Columns removed: 1


In [45]:
# This chunk enumerates features that utilise strings as values to prepare them for the LCS model.
# This chunk also provides a key for the enumerated values.

# Identify columns with string/object dtype
string_cols = df_clean.select_dtypes(include=['object']).columns

# Create a new enum column for each, suffixed with '_enum'
for col in string_cols:
    df_clean[col + '_enum'] = pd.factorize(df_clean[col])[0]
    mapping = df_clean[[col, col + '_enum']].drop_duplicates().sort_values(col + '_enum')
    print(f"\n{col} mapping:")
    print(mapping)

# Drop the original string columns, keeping their _enum replacements
df_clean = df_clean.drop(columns=string_cols)

# !!! It is important when setting up the model that these new features are defined as discrete. A float return would be meaningless!!!
# This can be done with 


sex mapping:
      sex  sex_enum
2  female         0
6    male         1

dzgroup mapping:
              dzgroup  dzgroup_enum
2           Cirrhosis             0
4         Lung Cancer             1
5   ARF/MOSF w/Sepsis             2
6                Coma             3
7                 CHF             4
25       Colon Cancer             5
26               COPD             6
28       MOSF w/Malig             7

dzclass mapping:
              dzclass  dzclass_enum
2  COPD/CHF/Cirrhosis             0
4              Cancer             1
5            ARF/MOSF             2
6                Coma             3

race mapping:
        race  race_enum
2      white          0
22     other          1
24     asian          2
32  hispanic          3
40     black          4

ca mapping:
            ca  ca_enum
2           no        0
4   metastatic        1
28         yes        2

sfdm2 mapping:
                   sfdm2  sfdm2_enum
2       <2 mo. follow-up           0
4    no(M2 and SIP pres)    

C:\Users\itsjo\AppData\Local\Temp\ipykernel_23496\1388001100.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_clean.select_dtypes(include=['object']).columns


In [46]:
# scikit already assumes that any int columns are to be kept discrete if it has under 10 int values. 
# This is a check to see if any adjustments need to be made from the default settings.
for col in df_clean.columns:
    if col.endswith('_enum'):
        print(col, df_clean[col].nunique())

sex_enum 2
dzgroup_enum 8
dzclass_enum 4
race_enum 5
ca_enum 3
sfdm2_enum 5


In [47]:
# Verifying changes made:

print("--- Missing values ---")
print(df_clean.isna().sum())  # Count of missing values per column

print("\n=== Data Types ===")
print(df_clean.dtypes)

print("--- Duplicate rows ---")
print(df_clean.duplicated().sum())  # Count of duplicate rows

print("--- Columns ---")
before = df_raw.shape[1]
after = df_clean.shape[1]
print(f"Originally: {before}")
print(f"Removed: {before - after}")
print(f"Remaining: {after}")

print("--- Rows ---")
before = df_raw.shape[0]
after = df_clean.shape[0]
print(f"Originally: {before}")
print(f"Removed: {before - after}")
print(f"Remaining: {after}")

--- Missing values ---
age             0
slos            0
num.co          0
scoma           0
charges         0
avtisst         0
sps             0
aps             0
hday            0
diabetes        0
dementia        0
meanbp          0
wblc            0
hrt             0
resp            0
temp            0
crea            0
sod             0
sex_enum        0
dzgroup_enum    0
dzclass_enum    0
race_enum       0
ca_enum         0
sfdm2_enum      0
dtype: int64

=== Data Types ===
age             float64
slos              int64
num.co            int64
scoma           float64
charges         float64
avtisst         float64
sps             float64
aps             float64
hday              int64
diabetes          int64
dementia          int64
meanbp          float64
wblc            float64
hrt             float64
resp            float64
temp            float64
crea            float64
sod             float64
sex_enum          int64
dzgroup_enum      int64
dzclass_enum      int64
race_enu

In [48]:
# Updates the split in training and testing data 

X = df_clean.drop(columns=['sfdm2_enum'])   
y = df_clean['sfdm2_enum']   

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [49]:
# Checking how bad the imbalance is on the new data frame

print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

sfdm2_enum
0    2317
1    2286
3     691
2     419
4      31
Name: count, dtype: int64
sfdm2_enum
0    40.337744
1    39.798050
3    12.029944
2     7.294568
4     0.539694
Name: proportion, dtype: float64


In [50]:
# Capping counts at 1000 to reduce imbalance.

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy={0: 1000, 1: 1000}, random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

In [51]:
print(y_train.value_counts())       # before
print(pd.Series(y_train_res).value_counts())  # after — should now be equal counts

sfdm2_enum
0    2317
1    2286
3     691
2     419
4      31
Name: count, dtype: int64
sfdm2_enum
0    1000
1    1000
3     691
2     419
4      31
Name: count, dtype: int64


There is still a very significant imbalance in this data. It may be worth synthesising some data for rarer cases to help balance the odds? Keen to hear yall's thoughts on this. 